In [30]:
!pip install sklearn-pandas pandas scikit-learn

In [31]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn_pandas import DataFrameMapper
import numpy as np

URL_BASE = "https://raw.githubusercontent.com/FranciscoAmbort/grupo13_castellano_ambort_ia_2026/main/entrega_3/data"

url_train = f"{URL_BASE}/solicitudes_train.csv"
url_test = f"{URL_BASE}/solicitudes_test.csv"
url_comp = f"{URL_BASE}/comportamiento_bancario.csv"

# 2. Carga directa optimizada
df_sol_train = pd.read_csv(url_train)
df_sol_test = pd.read_csv(url_test)
comportamiento = pd.read_csv(url_comp)

In [32]:
# Regresion logistica con modificacion del data set: Intentamos inferir patrones en los datos creando features nuevas. Este representa el codigo del intento 8_v2 el cual es el que mejor obtuvimos(159862 score)
# Eliminamos las columnas 'Dias_Atraso_Max','Monto_Solicitado','Tasa_Interes', 'Tiene_Historial' debido a que al entrenar el modelo, veaimos que los valores finales de sus coeficientes en la regresion logistica eran muy bajos, esto nos indica que
# no aportan en la decision logistica, simplemente hacen que tengamos columnas innecesarias.
# Ademas creamos Esfuerzo_Pago y Uso_Dinero (explicadas en informe)
# Otro cambio que implementamos fue agrupar los motivos de prestamos en 3 grupos segun los coeficientes observados. Este cambio pensamos que mejoraria el score en kaggle, sin embargo, se mantuvo/disminuyo muy poco.(159243.04)

In [33]:

df_compor_tratado = comportamiento.copy()
# Ordenamiento cronológico por usuario
df_compor_tratado["Mes_Referencia"] = pd.to_datetime(df_compor_tratado["Mes_Referencia"])
df_compor_tratado = df_compor_tratado.sort_values(
    by=["ID_Usuario", "Mes_Referencia"], ascending=[True, True]
)

# Métricas de atrasos de deuda
df_atrasos = (
    df_compor_tratado.groupby("ID_Usuario")["Dias_Atraso_Otras_Deudas"]
    .agg(Dias_Atraso_Max="max", Dias_Atraso_Mean="mean")
    .reset_index()
)

# Ventana temporal: promedio de saldo en los últimos 6 meses
ultimos_6m = df_compor_tratado.groupby("ID_Usuario").tail(6)
df_saldo_reciente = (
    ultimos_6m.groupby("ID_Usuario")["Saldo_Promedio_Mensual"]
    .agg(Saldo_Prom_Ultimos_6M="mean")
    .reset_index()
)

# Tabla agregada final de comportamiento
comportamiento_procesado = pd.merge(
    df_atrasos, df_saldo_reciente, on="ID_Usuario", how="inner"
)

# ==============================================================================
# 3. GENERACIÓN DE FEATURES BASE Y CRUCE CON SOLICITUDES
# ==============================================================================
df_sol_train_tratado = df_sol_train.drop(columns=['Fecha_Solicitud']).copy()
df_sol_test_tratado = df_sol_test.drop(columns=['Fecha_Solicitud']).copy()

# Feature Esfuerzo_Pago (Monto / Ingreso)
df_sol_train_tratado['Esfuerzo_Pago'] = (
    df_sol_train_tratado['Monto_Solicitado'] / (df_sol_train_tratado['Ingreso_Mensual'] + 1)
)
df_sol_test_tratado['Esfuerzo_Pago'] = (
    df_sol_test_tratado['Monto_Solicitado'] / (df_sol_test_tratado['Ingreso_Mensual'] + 1)
)

# Merge final hacia train_sol_comp_procesados
train_sol_comp_procesados = pd.merge(
    df_sol_train_tratado,
    comportamiento_procesado,
    on='ID_Usuario',
    how='left'
)
train_sol_comp_procesados['Tiene_Historial'] = (
    train_sol_comp_procesados['Saldo_Prom_Ultimos_6M'].notna().astype(int)
)

# Merge análogo para solicitudes de test
test_sol_comp_procesados = pd.merge(
    df_sol_test_tratado,
    comportamiento_procesado,
    on='ID_Usuario',
    how='left'
)
test_sol_comp_procesados['Tiene_Historial'] = (
    test_sol_comp_procesados['Saldo_Prom_Ultimos_6M'].notna().astype(int)
)

train_sol_comp_procesados.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Estado_Final,Esfuerzo_Pago,Dias_Atraso_Max,Dias_Atraso_Mean,Saldo_Prom_Ultimos_6M,Tiene_Historial
0,1,12698,7900,0.2333,4461,Consumo,1,1.770506,25,4.826087,3499.591667,1
1,2,20744,17400,0.2507,3889,Vivienda,1,4.473008,56,8.590909,6403.858333,1
2,3,6830,8600,0.2368,4035,Vehiculo,0,2.130823,57,19.590909,145.638333,1
3,4,5722,13300,0.2826,4018,Vehiculo,1,3.309281,53,3.043478,7458.051667,1
4,5,24719,6600,0.2314,3357,Otros,1,1.965456,32,2.478261,3016.070000,1


# Regresion Logistica EDA

In [34]:

train_mejorado_all= train_sol_comp_procesados.copy()
train_mejorado_all.head()

# 2. Columnas a borrar
columnas_a_borrar = [
    'Dias_Atraso_Max',
    'Monto_Solicitado',
    'Tasa_Interes',
    'Tiene_Historial',
]

# 3. Eliminar columnas
train_mejorado_all = train_mejorado_all.drop(columns=columnas_a_borrar)

# Nueva feature Uso_Dinero
train_mejorado_all['Uso_Dinero'] = (
    train_mejorado_all['Saldo_Prom_Ultimos_6M'] / (train_mejorado_all['Ingreso_Mensual'])
)

# Elimino ingreso y Saldo promedio ultimos 6m debido a que se utilizan en Uso_Dinero y observamos que al entrenar el modelo, sus coeficientes ahora en la regresion logistica eran bajos.
train_mejorado_all = train_mejorado_all.drop(columns=['Ingreso_Mensual'])
train_mejorado_all = train_mejorado_all.drop(columns=['Saldo_Prom_Ultimos_6M'])

mapper_lr_8_v2 = DataFrameMapper([
    (['Motivo_Prestamo'], [OneHotEncoder()]),
    (['Esfuerzo_Pago'], [StandardScaler()]),
    (['Dias_Atraso_Mean'], [StandardScaler()]),
    (['Uso_Dinero'], [StandardScaler()])
])

lr_model_procesado_8_v2 = Pipeline([
    ('mapper', mapper_lr_8_v2),
    ('classifier', LogisticRegression(
        random_state=42
    ))
])


lr_model_procesado_8_v2.fit(
    train_mejorado_all,
    train_mejorado_all['Estado_Final']
)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Pipeline(steps=[('mapper',
                 DataFrameMapper(drop_cols=[],
                                 features=[(['Motivo_Prestamo'],
                                            [OneHotEncoder()]),
                                           (['Esfuerzo_Pago'],
                                            [StandardScaler()]),
                                           (['Dias_Atraso_Mean'],
                                            [StandardScaler()]),
                                           (['Uso_Dinero'],
                                            [StandardScaler()])])),
                ('classifier', LogisticRegression(random_state=42))])

In [35]:
# Prediccion
test_eval_lr = test_sol_comp_procesados.copy()

# Agregamos las features que colocamos en validacion
test_eval_lr['Uso_Dinero'] = test_eval_lr['Saldo_Prom_Ultimos_6M'] / (
    test_eval_lr['Ingreso_Mensual'] + 1
)

test_eval_lr['prob_paga'] = lr_model_procesado_8_v2.predict_proba(test_eval_lr)[
    :, 1
]

candidatos_lr = test_eval_lr[test_eval_lr['prob_paga'] >= 0.5].sort_values(
    by='prob_paga',
    ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_lr = []

for idx, fila in candidatos_lr.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_lr.append(fila['ID_Solicitud'])
    else:
        continue

test_eval_lr['Decision_Aprobacion'] = test_eval_lr['ID_Solicitud'].isin(ids_aprobados_lr).astype(int)
submission_lr_nuevo = test_eval_lr[['ID_Solicitud', 'Decision_Aprobacion']]
submission_lr_nuevo.to_csv('submission_lr_feature_engineering_8_v2.csv', index=False)

print(f"Total registros: {len(submission_lr_nuevo)}")
print(f"Créditos aprobados: {submission_lr_nuevo['Decision_Aprobacion'].sum()}")
print(f"Monto total colocado: ${monto_acumulado:,.2f} USD")


Total registros: 3061
Créditos aprobados: 901
Monto total colocado: $2,000,000.00 USD


In [36]:

# 1. Agrupamos los motivos
def agrupar_motivos(motivo):
  motivo_limpio = str(motivo).strip()
  if motivo_limpio in ['Vivienda', 'Educacion', 'Salud']:
    return 'Alta_Confianza'
  elif motivo_limpio in ['Consumo', 'Vehiculo', 'Otros']:
    return 'Neutro'
  else:
    return 'Riesgoso'

# 2. Preparamos train
train_mejorado_all = train_sol_comp_procesados.copy()

columnas_a_borrar = [
    'Dias_Atraso_Max',
    'Monto_Solicitado',
    'Tasa_Interes',
    'Tiene_Historial',
]
train_mejorado_all = train_mejorado_all.drop(columns=columnas_a_borrar)

train_mejorado_all['Uso_Dinero'] = train_mejorado_all['Saldo_Prom_Ultimos_6M'] / (
    train_mejorado_all['Ingreso_Mensual'] + 1
)

train_mejorado_all['Motivo_Grupo'] = train_mejorado_all['Motivo_Prestamo'].apply(
    agrupar_motivos
)
train_mejorado_all = train_mejorado_all.drop(
    columns=['Motivo_Prestamo', 'Ingreso_Mensual', 'Saldo_Prom_Ultimos_6M']
)

mapper_lr_8_v4 = DataFrameMapper([
    (['Motivo_Grupo'], [OneHotEncoder()]),
    (['Esfuerzo_Pago'], [StandardScaler()]),
    (['Dias_Atraso_Mean'], [StandardScaler()]),
    (['Uso_Dinero'], [StandardScaler()]),
])

lr_model_procesado_8_v4 = Pipeline([
    ('mapper', mapper_lr_8_v4),
    ('classifier', LogisticRegression(random_state=42)),
])

lr_model_procesado_8_v4.fit(
    train_mejorado_all, train_mejorado_all['Estado_Final']
)

# 5. Preparamos test
test_eval_lr = test_sol_comp_procesados.copy()

test_eval_lr['Uso_Dinero'] = test_eval_lr['Saldo_Prom_Ultimos_6M'] / (
    test_eval_lr['Ingreso_Mensual'] + 1
)
test_eval_lr['Motivo_Grupo'] = test_eval_lr['Motivo_Prestamo'].apply(
    agrupar_motivos
)

test_eval_lr['prob_paga'] = lr_model_procesado_8_v4.predict_proba(test_eval_lr)[
    :, 1
]

candidatos_lr = test_eval_lr[test_eval_lr['prob_paga'] >= 0.5].sort_values(
    by='prob_paga', ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_lr = []

for idx, fila in candidatos_lr.iterrows():
  monto = fila['Monto_Solicitado']
  if monto_acumulado + monto <= PRESUPUESTO_MAX:
    monto_acumulado += monto
    ids_aprobados_lr.append(fila['ID_Solicitud'])
  else:
    continue

test_eval_lr['Decision_Aprobacion'] = (
    test_eval_lr['ID_Solicitud'].isin(ids_aprobados_lr).astype(int)
)
submission_lr_v4 = test_eval_lr[['ID_Solicitud', 'Decision_Aprobacion']]
submission_lr_v4.to_csv(
    'submission_lr_feature_engineering_8_v4.csv', index=False
)

print('\n--- RESUMEN DE ASIGNACIÓN ---')
print(f'Total registros: {len(submission_lr_v4)}')
print(f"Créditos aprobados: {submission_lr_v4['Decision_Aprobacion'].sum()}")
print(f'Monto total colocado: ${monto_acumulado:,.2f} USD')
if submission_lr_v4['Decision_Aprobacion'].sum() > 0:
  print(
      f'Monto promedio aprobado:'
      f' ${monto_acumulado / submission_lr_v4["Decision_Aprobacion"].sum():,.2f}'
      ' USD'
  )

/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)



--- RESUMEN DE ASIGNACIÓN ---
Total registros: 3061
Créditos aprobados: 908
Monto total colocado: $1,999,900.00 USD
Monto promedio aprobado: $2,202.53 USD
